# Social Distance Detection using YOLOv3: Colab walkthrough

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aish-19shanbhag/social-distance-detection/blob/main/notebooks/social_distance_walkthrough.ipynb)

This notebook clones the repository, downloads the pre-trained detectors, and runs the full pipeline on the sample pedestrian video:

1. detect people (YOLOv3 or SSD MobileNet, through OpenCV DNN)
2. map each person's foot point to a calibrated bird's-eye ground plane
3. measure pairwise distances in centimetres and classify each person as **high risk** (≤ 150 cm), **low risk** (150-180 cm) or **no risk** (> 180 cm)

Run everything with **Runtime > Run all**. A CPU runtime is enough.

## 1. Setup

In [ ]:
!git clone -q https://github.com/aish-19shanbhag/social-distance-detection.git
%cd social-distance-detection
!pip install -q -e .   # Colab already ships OpenCV 4.x

import sys
sys.path.insert(0, "src")  # make the package importable in this running kernel

## 2. Download the models

SSD MobileNet is about 23 MB. YOLOv3's weights are about 248 MB from pjreddie.com; if that download fails, the notebook falls back to SSD MobileNet.

In [ ]:
!python scripts/download_models.py

from socialdistance.detectors import models_available
DETECTOR = "yolov3" if models_available("yolov3", "models") else "ssd"
print("Using detector:", DETECTOR)

## 3. Look at the calibration

The 4 red points are the region of interest (a rectangle on the real ground); points 5-6 and 5-7 are 180 cm apart on the ground. The calibration for this sample video is approximate (see `config/vtest_calibration.json`).

In [ ]:
import cv2, json
import matplotlib.pyplot as plt
from socialdistance.calibration import load_calibration
from socialdistance.perspective import BirdEyeTransform
from socialdistance.visualize import draw_calibration

cal = load_calibration("config/vtest_calibration.json")
cap = cv2.VideoCapture("data/vtest.avi"); ok, frame = cap.read(); cap.release()
t = BirdEyeTransform(cal, frame.shape[1::-1])
print(f"ROI is about {t.roi_size_cm[0]/100:.1f} m x {t.roi_size_cm[1]/100:.1f} m on the ground")
plt.figure(figsize=(10, 7)); plt.imshow(cv2.cvtColor(draw_calibration(frame, t), cv2.COLOR_BGR2RGB)); plt.axis("off");

## 4. Run the pipeline on the sample video

In [ ]:
!python -m socialdistance run --video data/vtest.avi --config config/vtest_calibration.json \
    --detector $DETECTOR --max-frames 300 --output outputs/colab_run

## 5. Results: one frame and the bird's-eye view

In [ ]:
import pandas as pd
from socialdistance.detectors import create_detector
from socialdistance.pipeline import iter_video, render
from socialdistance.visualize import side_by_side

df = pd.read_csv("outputs/colab_run/risk_counts.csv")
busiest = int(df.loc[(df.high_risk + df.low_risk).idxmax(), "frame"])
det = create_detector(DETECTOR, "models")
for frame, result, transform in iter_video("data/vtest.avi", det, cal, start_frame=busiest, max_frames=1):
    annotated, bird = render(frame, result, transform, f"frame {busiest}")
plt.figure(figsize=(16, 8)); plt.imshow(cv2.cvtColor(side_by_side(annotated, bird), cv2.COLOR_BGR2RGB)); plt.axis("off")
print(result.counts, "closest pair (cm):", result.assessment.min_distance_cm)

## 6. Risk counts over time

In [ ]:
ax = df.set_index(df.frame / 10)[["high_risk", "low_risk", "no_risk"]].plot.area(
    figsize=(12, 3.5), color=["#d03b3b", "#fab219", "#0ca30c"], linewidth=0)
ax.set_xlabel("Video time (s)"); ax.set_ylabel("People in frame"); ax.set_title("People per risk zone")
print(json.dumps(json.load(open("outputs/colab_run/summary.json")), indent=2))

## 7. Demo GIF from the repository (generated with SSD MobileNet)

In [ ]:
from IPython.display import Image as IPImage
IPImage(filename="reports/demo/demo.gif")

The annotated video for this run is in `outputs/colab_run/annotated.mp4` (download it from the file browser on the left).

## 8. Try your own video

1. Upload a video, then calibrate it on a desktop with `python -m socialdistance calibrate --video my.mp4 --out config/my.json` (click 4 ground-rectangle corners, then 3 points 180 cm apart), or write the 7 points into a JSON file like `config/vtest_calibration.json`.
2. Run `python -m socialdistance run --video my.mp4 --config config/my.json --detector yolov3`.